In [1]:
import sys
import os
import pandas as pd
import pickle

# Get the absolute path of the project root directory
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

# Define the directory and file path
save_dir = "../data/parsed/"
load_path = os.path.join(save_dir, "builder.pkl")

# Load the builder object
with open(load_path, "rb") as f:
    builder = pickle.load(f)

In [2]:
sample = builder.run(builder.timestamps[0])
sample

This pandapower network includes the following parameter tables:
   - bus (118 elements)
   - load (91 elements)
   - gen (321 elements)
   - ext_grid (1 element)
   - line (177 elements)
   - trafo (9 elements)
   - bus_geodata (118 elements)
 and the following results tables:
   - res_bus (118 elements)
   - res_line (177 elements)
   - res_trafo (9 elements)
   - res_ext_grid (1 element)
   - res_load (91 elements)
   - res_gen (321 elements)

In [ ]:
import networkx as nx
import torch
import torch.nn as nn
import torch_geometric.nn as pyg_nn
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx

def construct_nrel118_graph_sequence(builder, num_timesteps=100):
    """
    Constructs a sequence of PyTorch Geometric Data objects for the NREL 118-bus network.
    
    Args:
        builder: PandaPowerFlowBuilder instance
        num_timesteps: Number of timesteps to process
    
    Returns:
        List of PyG Data objects representing the power network state at each timestep
    """
    graph_sequence = []
    
    for t in range(min(num_timesteps, len(builder.timestamps))):
        sample = builder.run(builder.timestamps[t])
        G = nx.Graph()
        
        # Add nodes (buses) with their features
        for idx, bus in sample.res_bus.iterrows():
            G.add_node(idx, 
                      vm_pu=float(bus['vm_pu']),
                      va_degree=float(bus['va_degree']),
                      p_mw=float(bus['p_mw']),
                      q_mvar=float(bus['q_mvar']))
        
        # Add edges (transmission lines)
        for idx, line in sample.res_line.iterrows():
            from_bus = sample.line.loc[idx, 'from_bus']
            to_bus = sample.line.loc[idx, 'to_bus']
            G.add_edge(from_bus, to_bus,
                      p_from=float(line['p_from_mw']),
                      q_from=float(line['q_from_mvar']),
                      p_to=float(line['p_to_mw']),
                      q_to=float(line['q_to_mvar']),
                      loading=float(line['loading_percent']))
        
        # Add transformers as edges
        for idx, trafo in sample.res_trafo.iterrows():
            hv_bus = sample.trafo.loc[idx, 'hv_bus']
            lv_bus = sample.trafo.loc[idx, 'lv_bus']
            G.add_edge(hv_bus, lv_bus,
                      p_from=float(trafo['p_hv_mw']),
                      q_from=float(trafo['q_hv_mvar']),
                      p_to=float(trafo['p_lv_mw']),
                      q_to=float(trafo['q_lv_mvar']),
                      loading=float(trafo['loading_percent']))
        
        # Convert to PyG Data
        data = from_networkx(G)
        
        # Extract node features
        node_features = torch.tensor([[G.nodes[i]['vm_pu'], 
                                       G.nodes[i]['va_degree'],
                                       G.nodes[i]['p_mw'], 
                                       G.nodes[i]['q_mvar']] 
                                      for i in G.nodes], 
                                     dtype=torch.float)
        
        # Ensure bidirectional edges match edge_index.shape[1]
        edge_features_list = []
        for u, v, d in G.edges(data=True):
            edge_features_list.append([d['p_from'], d['q_from'], d['p_to'], d['q_to'], d['loading']])
            edge_features_list.append([d['p_to'], d['q_to'], d['p_from'], d['q_from'], d['loading']])  # Duplicate

        edge_features = torch.tensor(edge_features_list, dtype=torch.float)
        
        data.x = node_features
        data.edge_attr = edge_features
        
        graph_sequence.append(data)
    
    return graph_sequence